# 02 — Cleaning & Feature Engineering

Chuẩn hóa dữ liệu, tạo biến phân tích, segment người dùng, xuất processed data cho Streamlit và Power BI.

In [1]:

from pathlib import Path
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 120)

# Paths
PROJECT_ROOT = Path("..")
RAW_PATH = PROJECT_ROOT / "data" / "raw" / "mobile_usage_behavioral_analysis.csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
AGG_DIR = PROJECT_ROOT / "data" / "aggregated"
FIGURE_DIR = PROJECT_ROOT / "reports" / "figures"

# Create directories if they don't exist
for p in [PROCESSED_DIR, AGG_DIR, FIGURE_DIR]:
    p.mkdir(parents=True, exist_ok=True)
def title(t):
    print("\n" + "="*90); print(t); print("="*90)
def clean_cols(df):
    df = df.copy(); df.columns = [c.strip() for c in df.columns]; return df

# Load and clean dataset
title("Load and clean raw dataset")
df = clean_cols(pd.read_csv(RAW_PATH)).drop_duplicates().copy()
num_cols = ["Age", "Total_App_Usage_Hours", "Daily_Screen_Time_Hours", "Number_of_Apps_Used", "Social_Media_Usage_Hours", "Productivity_App_Usage_Hours", "Gaming_App_Usage_Hours"]
cat_cols = ["User_ID", "Gender", "Location"]

for c in num_cols:
    df[c] = pd.to_numeric(df[c], errors="coerce")
for c in cat_cols:
    df[c] = df[c].astype(str).str.strip()

invalid = (df["Age"] < 0) | (df["Daily_Screen_Time_Hours"] < 0) | (df["Total_App_Usage_Hours"] < 0) | (df["Number_of_Apps_Used"] < 0)
print(f"Invalid rows removed: {invalid.sum()}")
df = df.loc[~invalid].copy()
print(df.shape)
display(df.head())



Load and clean raw dataset
Invalid rows removed: 0
(1000, 10)


,User_ID,Age,Gender,Total_App_Usage_Hours,Daily_Screen_Time_Hours,Number_of_Apps_Used,Social_Media_Usage_Hours,Productivity_App_Usage_Hours,Gaming_App_Usage_Hours,Location
0,1,56,Male,2.61,7.15,24,4.43,0.55,2.40,Los Angeles
1,2,46,Male,2.13,13.79,18,4.67,4.42,2.43,Chicago
2,3,32,Female,7.28,4.50,11,4.58,1.71,2.83,Houston
3,4,25,Female,1.20,6.29,21,3.18,3.42,4.58,Phoenix
4,5,38,Male,6.31,12.59,14,3.15,0.13,4.00,New York


In [2]:
# Feature engineering
title("Feature engineering")
def age_group(age):
    if age <= 24: return "18–24"
    if age <= 34: return "25–34"
    if age <= 44: return "35–44"
    if age <= 54: return "45–54"
    return "55–59"

def screen_segment(h):
    if h < 4: return "Light (<4h)"
    if h < 8: return "Moderate (4–8h)"
    if h < 12: return "Heavy (8–12h)"
    return "Extreme (>12h)"

def dominant_lifestyle(row):
    scores = {
        "Social Enthusiast": row["Social_Media_Usage_Hours"],
        "Productivity Focused": row["Productivity_App_Usage_Hours"],
        "Mobile Gamer": row["Gaming_App_Usage_Hours"],
    }
    return max(scores, key=scores.get)

# Create new features
df["Age_Group"] = df["Age"].apply(age_group)
df["Screen_Time_Segment"] = df["Daily_Screen_Time_Hours"].apply(screen_segment)
df["Dominant_Lifestyle"] = df.apply(dominant_lifestyle, axis=1)
df["Entertainment_Hours"] = df["Social_Media_Usage_Hours"] + df["Gaming_App_Usage_Hours"]
df["Category_Usage_Sum"] = df["Social_Media_Usage_Hours"] + df["Productivity_App_Usage_Hours"] + df["Gaming_App_Usage_Hours"]
df["Screen_to_App_Gap"] = df["Daily_Screen_Time_Hours"] - df["Total_App_Usage_Hours"]
df["App_Fragmentation_Score"] = df["Total_App_Usage_Hours"] / df["Number_of_Apps_Used"].replace(0, np.nan)
df["Social_Share_of_Category_Usage"] = df["Social_Media_Usage_Hours"] / df["Category_Usage_Sum"].replace(0, np.nan)
df["Productivity_Share_of_Category_Usage"] = df["Productivity_App_Usage_Hours"] / df["Category_Usage_Sum"].replace(0, np.nan)
df["Gaming_Share_of_Category_Usage"] = df["Gaming_App_Usage_Hours"] / df["Category_Usage_Sum"].replace(0, np.nan)

display(df.head())



Feature engineering


,User_ID,Age,Gender,Total_App_Usage_Hours,Daily_Screen_Time_Hours,Number_of_Apps_Used,Social_Media_Usage_Hours,Productivity_App_Usage_Hours,Gaming_App_Usage_Hours,Location,Age_Group,Screen_Time_Segment,Dominant_Lifestyle,Entertainment_Hours,Category_Usage_Sum,Screen_to_App_Gap,App_Fragmentation_Score,Social_Share_of_Category_Usage,Productivity_Share_of_Category_Usage,Gaming_Share_of_Category_Usage
0,1,56,Male,2.61,7.15,24,4.43,0.55,2.40,Los Angeles,55–59,Moderate (4–8h),Social Enthusiast,6.83,7.38,4.54,0.108750,0.600271,0.074526,0.325203
1,2,46,Male,2.13,13.79,18,4.67,4.42,2.43,Chicago,45–54,Extreme (>12h),Social Enthusiast,7.10,11.52,11.66,0.118333,0.405382,0.383681,0.210938
2,3,32,Female,7.28,4.50,11,4.58,1.71,2.83,Houston,25–34,Moderate (4–8h),Social Enthusiast,7.41,9.12,-2.78,0.661818,0.502193,0.187500,0.310307
3,4,25,Female,1.20,6.29,21,3.18,3.42,4.58,Phoenix,25–34,Moderate (4–8h),Mobile Gamer,7.76,11.18,5.09,0.057143,0.284436,0.305903,0.409660
4,5,38,Male,6.31,12.59,14,3.15,0.13,4.00,New York,35–44,Extreme (>12h),Mobile Gamer,7.15,7.28,6.28,0.450714,0.432692,0.017857,0.549451


In [3]:
# Validate engineered features
title("Validate engineered features")
feature_dictionary = pd.DataFrame([
    ["Age_Group", "Faceting and demographic comparison"],
    ["Screen_Time_Segment", "Screen-time severity segmentation"],
    ["Dominant_Lifestyle", "Main lifestyle category per user"],
    ["Entertainment_Hours", "Social + gaming behavioral intensity"],
    ["Category_Usage_Sum", "Sum of category indicators; not exact total usage"],
    ["Screen_to_App_Gap", "Gap between screen time and app usage"],
    ["App_Fragmentation_Score", "Average usage per app used"],
])

feature_dictionary.columns = ["Feature", "Purpose"]
display(feature_dictionary)
display(df["Age_Group"].value_counts().sort_index().to_frame("Users"))
display(df["Screen_Time_Segment"].value_counts().to_frame("Users"))
display(df["Dominant_Lifestyle"].value_counts().to_frame("Users"))
fig = px.bar(df["Screen_Time_Segment"].value_counts().reset_index(), x="count", y="Screen_Time_Segment", orientation="h", title="Screen time segment distribution")
fig.show()



Validate engineered features


,Feature,Purpose
0,Age_Group,Faceting and demographic comparison
1,Screen_Time_Segment,Screen-time severity segmentation
2,Dominant_Lifestyle,Main lifestyle category per user
3,Entertainment_Hours,Social + gaming behavioral intensity
4,Category_Usage_Sum,Sum of category indicators; not exact total usage
5,Screen_to_App_Gap,Gap between screen time and app usage
6,App_Fragmentation_Score,Average usage per app used


,Users
Age_Group,
18–24,169
25–34,225
35–44,232
45–54,265
55–59,109


,Users
Screen_Time_Segment,
Heavy (8–12h),331
Moderate (4–8h),295
Light (<4h),217
Extreme (>12h),157


,Users
Dominant_Lifestyle,
Productivity Focused,349
Social Enthusiast,338
Mobile Gamer,313


In [4]:
# Create aggregate tables
title("Create aggregate tables")
kpi_summary = pd.DataFrame({
    "Metric": ["Total Users", "Average Age", "Average Daily Screen Time", "Average Total App Usage", "Average Apps Used", "Heavy/Extreme User Rate"],
    "Value": [len(df), df["Age"].mean(), df["Daily_Screen_Time_Hours"].mean(), df["Total_App_Usage_Hours"].mean(), df["Number_of_Apps_Used"].mean(), df[df["Screen_Time_Segment"].isin(["Heavy (8–12h)", "Extreme (>12h)"])].shape[0] / len(df)]
})

# Group by demographic and behavioral segments
age_summary = df.groupby("Age_Group", as_index=False)[num_cols].mean()
gender_summary = df.groupby("Gender", as_index=False)[num_cols].mean()
location_summary = df.groupby("Location", as_index=False)[num_cols].mean()
lifestyle_summary = df.groupby("Dominant_Lifestyle", as_index=False)[num_cols].mean()
segment_summary = df["Screen_Time_Segment"].value_counts().reset_index()
segment_summary.columns = ["Screen_Time_Segment", "Users"]
segment_summary["Percentage"] = segment_summary["Users"] / len(df)
for name, table in [("KPI", kpi_summary), ("Age", age_summary), ("Gender", gender_summary), ("Location", location_summary), ("Lifestyle", lifestyle_summary), ("Segment", segment_summary)]:
    print(f"\n{name} summary")
    display(table.round(3))



Create aggregate tables

KPI summary


,Metric,Value
0,Total Users,1000.000
1,Average Age,38.745
2,Average Daily Screen Time,7.696
3,Average Total App Usage,6.406
4,Average Apps Used,16.647
5,Heavy/Extreme User Rate,0.488



Age summary


,Age_Group,Age,Total_App_Usage_Hours,Daily_Screen_Time_Hours,Number_of_Apps_Used,Social_Media_Usage_Hours,Productivity_App_Usage_Hours,Gaming_App_Usage_Hours
0,18–24,20.787,6.558,8.047,16.260,2.564,2.457,2.427
1,25–34,29.462,6.252,7.568,17.240,2.481,2.368,2.486
2,35–44,39.849,6.402,7.646,16.172,2.254,2.695,2.605
3,45–54,49.645,6.437,7.637,16.536,2.487,2.509,2.306
4,55–59,56.899,6.419,7.671,17.303,2.595,2.361,2.663



Gender summary


...

,Location,Age,Total_App_Usage_Hours,Daily_Screen_Time_Hours,Number_of_Apps_Used,Social_Media_Usage_Hours,Productivity_App_Usage_Hours,Gaming_App_Usage_Hours
0,Chicago,39.568,5.978,8.137,17.198,2.388,2.418,2.504
1,Houston,37.193,6.343,7.453,15.928,2.551,2.497,2.456
2,Los Angeles,40.946,6.462,7.599,17.400,2.487,2.581,2.464
3,New York,37.313,6.610,7.397,16.407,2.386,2.421,2.483
4,Phoenix,39.065,6.573,7.949,16.362,2.494,2.578,2.466



Lifestyle summary


,Dominant_Lifestyle,Age,Total_App_Usage_Hours,Daily_Screen_Time_Hours,Number_of_Apps_Used,Social_Media_Usage_Hours,Productivity_App_Usage_Hours,Gaming_App_Usage_Hours
0,Mobile Gamer,38.460,6.087,7.491,16.581,1.812,1.916,3.752
1,Productivity Focused,39.461,6.494,7.898,16.430,1.762,3.747,1.856
2,Social Enthusiast,38.269,6.609,7.678,16.932,3.770,1.739,1.932



Segment summary


,Screen_Time_Segment,Users,Percentage
0,Heavy (8–12h),331,0.331
1,Moderate (4–8h),295,0.295
2,Light (<4h),217,0.217
3,Extreme (>12h),157,0.157


In [5]:
# Export processed datasets
title("Export processed datasets")
df.to_csv(PROCESSED_DIR / "smartphone_features.csv", index=False)
df.to_csv(PROCESSED_DIR / "smartphone_powerbi_model.csv", index=False)
kpi_summary.to_csv(AGG_DIR / "kpi_summary.csv", index=False)
age_summary.to_csv(AGG_DIR / "age_group_summary.csv", index=False)
gender_summary.to_csv(AGG_DIR / "gender_summary.csv", index=False)
location_summary.to_csv(AGG_DIR / "location_summary.csv", index=False)
lifestyle_summary.to_csv(AGG_DIR / "lifestyle_summary.csv", index=False)
segment_summary.to_csv(AGG_DIR / "screen_time_segment_summary.csv", index=False)
print("Export completed.")



Export processed datasets
Export completed.
